## Inspect the zone lookup ##

In [0]:

from pyspark.sql import functions as F

# Location of the official zone lookup
zone_path = "/Volumes/workspace/default/raw_landing/taxi_zone_lookup.csv"

# Read the CSV with its header
zones_raw = (
    spark.read
    .option("header", True)
    .csv(zone_path)
)

# Inspect the data
display(zones_raw.limit(10))

print("Total zones:", zones_raw.count())
print("Columns:", zones_raw.columns)


In [0]:

# Convert LocationID into an integer
zones_checked = zones_raw.withColumn(
    "location_id",
    F.col("LocationID").cast("int")
)

# Check for missing or invalid location IDs
missing_ids = zones_checked.filter(
    F.col("location_id").isNull()
).count()

# Check for duplicate location IDs
duplicate_ids = (
    zones_checked
    .groupBy("location_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Missing/invalid IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids.count())

display(duplicate_ids)


### Build gold.dim_zone ###

In [0]:

from pyspark.sql import functions as F

# 1. Create the Gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# 2. Standardize the taxi-zone dimension
dim_zone = zones_checked.select(
    F.col("location_id").alias("zone_key"),
    F.col("location_id"),
    F.trim(F.col("Borough")).alias("borough"),
    F.trim(F.col("Zone")).alias("zone_name"),
    F.trim(F.col("service_zone")).alias("service_zone"),
    F.lit(False).alias("is_unknown")
)

# 3. Create a fallback member for missing/unmatched zones
unknown_zone = spark.range(1).select(
    F.lit(-1).cast("int").alias("zone_key"),
    F.lit(None).cast("int").alias("location_id"),
    F.lit("Unknown").alias("borough"),
    F.lit("Unmapped or missing").alias("zone_name"),
    F.lit("Unknown").alias("service_zone"),
    F.lit(True).alias("is_unknown")
)

# 4. Combine real zones with the unknown member
dim_zone = dim_zone.unionByName(unknown_zone)

# 5. Validate before saving
assert dim_zone.count() == 266
assert dim_zone.select("zone_key").distinct().count() == 266

# 6. Save the Gold dimension as a Delta table
(
    dim_zone.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_zone")
)

# 7. Verify the saved table
saved_dim = spark.table("workspace.gold.dim_zone")

print("Dimension rows:", saved_dim.count())
print("Unique keys:", saved_dim.select("zone_key").distinct().count())
print("Gold dim_zone created successfully.")

display(saved_dim.orderBy("zone_key").limit(10))


### Build gold.dim_date ###

In [0]:

# Create one row for every day in January 2025
dim_date = spark.sql("""
    SELECT
        CAST(date_format(full_date, 'yyyyMMdd') AS INT) AS date_key,
        full_date,
        year(full_date) AS year,
        quarter(full_date) AS quarter,
        month(full_date) AS month,
        dayofmonth(full_date) AS day_of_month,
        date_format(full_date, 'EEEE') AS day_of_week,
        dayofweek(full_date) IN (1, 7) AS is_weekend
    FROM (
        SELECT explode(
            sequence(
                DATE '2025-01-01',
                DATE '2025-01-31',
                INTERVAL 1 DAY
            )
        ) AS full_date
    )
""")

# Validate the dimension
assert dim_date.count() == 31
assert dim_date.select("date_key").distinct().count() == 31

# Save as a managed Delta table
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.dim_date")
)

# Verify saved records
saved_date = spark.table("workspace.gold.dim_date")

print("Date dimension rows:", saved_date.count())
print("Unique date keys:", saved_date.select("date_key").distinct().count())

display(saved_date.orderBy("full_date"))


### Check which Silver records belong to January ###

In [0]:

from pyspark.sql import functions as F

# Read the validated Silver records
silver_df = spark.table("workspace.silver.yellow_taxi_trips")

# Extract the calendar date from the pickup timestamp
pickup_date = F.to_date("tpep_pickup_datetime")

# Classify records based on their source month
silver_scoped = silver_df.withColumn(
    "month_status",
    F.when(
        (F.col("_source_month") == "2025-01") &
        pickup_date.between("2025-01-01", "2025-01-31"),
        "IN_SCOPE"
    ).otherwise("OUTSIDE_SOURCE_MONTH")
)

# Display counts for each classification
display(
    silver_scoped.groupBy("month_status").count()
)

# Verify that no records were lost
silver_count = silver_df.count()
classified_count = silver_scoped.count()

assert silver_count == classified_count

print("Silver records:", silver_count)
print("Classified records:", classified_count)
print("PASS: All Silver records classified")


### Join Silver records to our dimensions ###

In [0]:

from pyspark.sql import functions as F

# Only January records qualify for the fact
in_scope = silver_scoped.filter(
    F.col("month_status") == "IN_SCOPE"
).withColumn(
    "pickup_date_key",
    F.date_format("tpep_pickup_datetime", "yyyyMMdd").cast("int")
)

# Load our Gold dimensions
zones = spark.table("workspace.gold.dim_zone")
dates = spark.table("workspace.gold.dim_date")

# Prepare separate references to the same zone dimension
pickup_zones = zones.select(
    F.col("location_id").alias("p_location_id"),
    F.col("zone_key").alias("p_zone_key")
)

dropoff_zones = zones.select(
    F.col("location_id").alias("d_location_id"),
    F.col("zone_key").alias("d_zone_key")
)

date_keys = dates.select(
    F.col("date_key").alias("matched_date_key")
)

# Join each trip to its date, pickup zone and drop-off zone
fact_joined = (
    in_scope
    .join(
        date_keys,
        F.col("pickup_date_key") == F.col("matched_date_key"),
        "left"
    )
    .join(
        pickup_zones,
        F.col("PULocationID") == F.col("p_location_id"),
        "left"
    )
    .join(
        dropoff_zones,
        F.col("DOLocationID") == F.col("d_location_id"),
        "left"
    )
    .withColumn(
        "pickup_zone_key",
        F.coalesce(F.col("p_zone_key"), F.lit(-1))
    )
    .withColumn(
        "dropoff_zone_key",
        F.coalesce(F.col("d_zone_key"), F.lit(-1))
    )
)

print("Dimension joins prepared.")


### Verify join integrity ###

In [0]:

join_audit = fact_joined.agg(
    F.count("*").alias("joined_rows"),

    F.sum(
        F.col("matched_date_key").isNull().cast("long")
    ).alias("unmatched_dates"),

    F.sum(
        (F.col("pickup_zone_key") == -1).cast("long")
    ).alias("unknown_pickup_zones"),

    F.sum(
        (F.col("dropoff_zone_key") == -1).cast("long")
    ).alias("unknown_dropoff_zones")
)

display(join_audit)

# Validate row count and date coverage
result = join_audit.first()

assert result["joined_rows"] == 3475080
assert result["unmatched_dates"] == 0

print("PASS: No records duplicated or lost in joins")
print("PASS: All pickup dates match dim_date")


#### Save the out-of-scope records ####

In [0]:

# Preserve records excluded from January Gold reporting
out_of_scope = (
    silver_scoped
    .filter(F.col("month_status") == "OUTSIDE_SOURCE_MONTH")
    .withColumn("quality_reason", F.lit("OUTSIDE_SOURCE_MONTH"))
)

(
    out_of_scope.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.yellow_taxi_out_of_scope")
)

print("Out-of-scope records:", out_of_scope.count())


#### Create the Gold fact table ####

In [0]:

# Select the business-ready fact table columns
fact_trips = fact_joined.select(

    # Dimension keys
    "pickup_date_key",
    "pickup_zone_key",
    "dropoff_zone_key",

    # Trip information
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.hour("tpep_pickup_datetime").alias("pickup_hour"),

    # Business attributes
    F.col("VendorID").alias("vendor_id"),
    "payment_type",
    "passenger_count",

    # Measurements
    F.col("trip_distance").alias("trip_distance_miles"),
    "trip_duration_minutes",
    "fare_amount",
    "total_amount",
    "tip_amount",

    # Data-quality indicators
    "is_zero_distance",
    "is_negative_fare",

    # Source lineage
    F.col("_source_month").alias("source_month"),
    F.col("_source_file").alias("source_file")
)

# Save the Gold fact table
(
    fact_trips.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.fact_yellow_trip")
)

print("Gold fact table created successfully.")


#### Verify the Gold fact table ####

In [0]:

# Load persisted tables
fact_saved = spark.table("workspace.gold.fact_yellow_trip")

bronze_count = spark.table(
    "workspace.bronze.yellow_taxi_trips"
).count()

fact_count = fact_saved.count()

quarantine_count = spark.table(
    "workspace.silver.yellow_taxi_quarantine"
).count()

excluded_count = spark.table(
    "workspace.silver.yellow_taxi_out_of_scope"
).count()

# Verify the fact table and complete source reconciliation
assert fact_count == 3475080
assert bronze_count == fact_count + quarantine_count + excluded_count

print("Gold fact records:", fact_count)
print("Quarantined records:", quarantine_count)
print("Outside-source-month records:", excluded_count)
print("Reconciliation difference:",
      bronze_count - fact_count - quarantine_count - excluded_count)

print("PASS: Gold fact created and source counts reconciled")

display(fact_saved.limit(5))


#### Verify Silver and Gold financial totals ####

In [0]:

from pyspark.sql import functions as F

# Select only the accepted January records
silver_january = (
    spark.table("workspace.silver.yellow_taxi_trips")
    .filter(F.col("_source_month") == "2025-01")
    .filter(
        F.to_date("tpep_pickup_datetime")
        .between("2025-01-01", "2025-01-31")
    )
)

gold_trips = spark.table("workspace.gold.fact_yellow_trip")

# Use decimal values for financial reconciliation
def financial_summary(df):
    return df.agg(
        F.count("*").alias("trip_count"),
        F.sum(
            F.col("fare_amount").cast("decimal(18,2)")
        ).alias("total_fares"),
        F.sum(
            F.col("total_amount").cast("decimal(18,2)")
        ).alias("total_charges")
    ).first()

silver_metrics = financial_summary(silver_january)
gold_metrics = financial_summary(gold_trips)

print("Silver:", silver_metrics.asDict())
print("Gold:", gold_metrics.asDict())

# Verify financial and record-count integrity
assert silver_metrics == gold_metrics

print("PASS: Trip counts and financial totals reconcile")


### Create our monthly KPI data mart ###

In [0]:

# Create business KPIs using Spark SQL
monthly_kpi = spark.sql("""
    SELECT
        source_month,

        COUNT(*) AS accepted_trip_count,

        ROUND(
            SUM(CAST(total_amount AS DECIMAL(18,2))), 2
        ) AS total_recorded_charges,

        ROUND(
            SUM(CAST(fare_amount AS DECIMAL(18,2))), 2
        ) AS total_fares,

        ROUND(AVG(trip_distance_miles), 2)
            AS avg_trip_distance,

        ROUND(AVG(trip_duration_minutes), 2)
            AS avg_trip_duration,

        SUM(CASE WHEN is_zero_distance THEN 1 ELSE 0 END)
            AS zero_distance_trips,

        SUM(CASE WHEN is_negative_fare THEN 1 ELSE 0 END)
            AS negative_fare_trips

    FROM workspace.gold.fact_yellow_trip
    GROUP BY source_month
""")

display(monthly_kpi)

# Validate before publishing
assert monthly_kpi.count() == 1
assert monthly_kpi.first()["accepted_trip_count"] == gold_metrics["trip_count"]

# Save as a managed Delta table
(
    monthly_kpi.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.agg_monthly_kpi")
)

print("PASS: Monthly KPI data mart created")


### Verify the persisted KPI table ###

In [0]:

saved_kpi = spark.table("workspace.gold.agg_monthly_kpi")

display(saved_kpi)

assert saved_kpi.count() == 1
assert saved_kpi.first()["accepted_trip_count"] == 3475080

print("PASS: Gold monthly KPI table verified")


### Create the daily pickup-zone data mart ###

In [0]:

from pyspark.sql import functions as F

# Aggregate trips by pickup date and zone
daily_zone = spark.sql("""
    SELECT
        pickup_date_key,
        pickup_zone_key,

        COUNT(*) AS accepted_trip_count,

        SUM(CAST(total_amount AS DECIMAL(18,2)))
            AS total_recorded_charges,

        SUM(CAST(fare_amount AS DECIMAL(18,2)))
            AS total_fares,

        ROUND(AVG(trip_distance_miles), 2)
            AS avg_trip_distance_miles,

        ROUND(AVG(trip_duration_minutes), 2)
            AS avg_trip_duration_minutes

    FROM workspace.gold.fact_yellow_trip

    GROUP BY
        pickup_date_key,
        pickup_zone_key
""")

# Inspect the result
print("Daily zone aggregate rows:", daily_zone.count())
display(daily_zone.limit(10))


### Reconcile the aggregate with the fact table  ###

In [0]:

fact = spark.table("workspace.gold.fact_yellow_trip")

# Independent totals from the fact table
fact_check = fact.agg(
    F.count("*").alias("trips"),
    F.sum(F.col("total_amount").cast("decimal(18,2)")).alias("charges"),
    F.sum(F.col("fare_amount").cast("decimal(18,2)")).alias("fares")
).first()

# Reconstruct totals from daily aggregates
mart_check = daily_zone.agg(
    F.sum("accepted_trip_count").alias("trips"),
    F.sum("total_recorded_charges").alias("charges"),
    F.sum("total_fares").alias("fares")
).first()

print("Fact totals:", fact_check.asDict())
print("Daily mart totals:", mart_check.asDict())

# Verify no records or monetary amounts were lost
assert fact_check.asDict() == mart_check.asDict()

print("PASS: Daily mart reconciles with Gold fact")


### Save the daily data mart ###

In [0]:

(
    daily_zone.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.agg_daily_pickup_zone")
)

saved_daily = spark.table(
    "workspace.gold.agg_daily_pickup_zone"
)

assert saved_daily.count() == daily_zone.count()

print("PASS: Daily pickup-zone mart saved")
print("Saved rows:", saved_daily.count())


### View the business results ###

In [0]:

# Top 10 pickup zones by accepted trip volume
top_zones = spark.sql("""
    SELECT
        z.borough,
        z.zone_name,
        SUM(d.accepted_trip_count) AS total_trips,
        SUM(d.total_recorded_charges) AS recorded_charges

    FROM workspace.gold.agg_daily_pickup_zone d

    JOIN workspace.gold.dim_zone z
        ON d.pickup_zone_key = z.zone_key

    GROUP BY z.borough, z.zone_name

    ORDER BY total_trips DESC

    LIMIT 10
""")

display(top_zones)
